In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [4]:
## Load the dataset
data = pd.read_csv("Churn_Modelling.csv")

In [5]:
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
#Preprocess the data: 1. Drop unnecessary columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)


In [8]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [9]:
#Encode categorical variables
label_encoder = LabelEncoder()
data['Gender'] = label_encoder.fit_transform(data['Gender'])

In [ ]:
data.head()

In [ ]:
data['Geography'].value_counts() #3 values, we can use one-hot encoding for this categorical variable

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [13]:
#Onehot encoding for Geography
from sklearn.preprocessing import OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False)
geography_encoded = onehot_encoder.fit_transform(data[['Geography']])

In [ ]:
onehot_encoder.get_feature_names_out[['Geography']]


In [15]:
geography_encoded


array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [16]:
geography_encoded_df = pd.DataFrame(geography_encoded, columns=onehot_encoder.get_feature_names_out(['Geography']))

In [ ]:
geography_encoded_df


In [14]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [18]:
#combine the new one-hot encoded columns with the original dataframe
data = data.drop('Geography', axis=1) #drop the original Geography column
data = pd.concat([data, geography_encoded_df], axis=1)

In [19]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [20]:
# Save preprocessed model to pickles
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder, file)

with open('onehot_encoder.pkl', 'wb') as file:
    pickle.dump(onehot_encoder, file)

Now divide into Independent and Target

In [21]:
X = data.drop('Exited' , axis=1)
y=data['Exited']

In [22]:
#test-train-split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [23]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((8000, 12), (2000, 12), (8000,), (2000,))

In [25]:
#Scaling the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [26]:
X_train_scaled, X_test_scaled

(array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
         -0.57946723, -0.57638802],
        [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
          1.72572313, -0.57638802],
        [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
         -0.57946723,  1.73494238],
        ...,
        [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
         -0.57946723, -0.57638802],
        [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
         -0.57946723, -0.57638802],
        [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
          1.72572313, -0.57638802]]),
 array([[-0.57749609,  0.91324755, -0.6557859 , ..., -0.99850112,
          1.72572313, -0.57638802],
        [-0.29729735,  0.91324755,  0.3900109 , ...,  1.00150113,
         -0.57946723, -0.57638802],
        [-0.52560743, -1.09499335,  0.48508334, ..., -0.99850112,
         -0.57946723,  1.73494238],
        ...,
        [ 0.81311987, -1.09499335,  0.77030065, ...,  

In [27]:
#Save as pickle (scaler model)
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

#NOW Training the model

In [28]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [33]:
#Building the ANN model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)), #first HL with input connection
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [35]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [36]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()

In [40]:
#compile the model
model.compile(optimizer =opt, loss = loss, metrics =['accuracy'])

In [ ]:
# Setup TensorBoard - callback for visualisation
import datetime
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorflow_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)

In [ ]:
# Setting up Early Stopping - for epoch stoppage
early_stopping_callback = EarlyStopping(monitor= 'val_loss', patience = 10, restore_best_weights=True)

In [53]:
# Now training the model
history = model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test), epochs = 100,callbacks =[tensorflow_callback, early_stopping_callback]
                    )

Epoch 1/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3186 - accuracy: 0.8677 - val_loss: 0.3567 - val_accuracy: 0.8540
Epoch 2/100
250/250 [==============================] - 0s 953us/step - loss: 0.3169 - accuracy: 0.8690 - val_loss: 0.3519 - val_accuracy: 0.8610
Epoch 3/100
250/250 [==============================] - 0s 946us/step - loss: 0.3159 - accuracy: 0.8683 - val_loss: 0.3632 - val_accuracy: 0.8550
Epoch 4/100
250/250 [==============================] - 0s 944us/step - loss: 0.3129 - accuracy: 0.8686 - val_loss: 0.3664 - val_accuracy: 0.8540
Epoch 5/100
250/250 [==============================] - 0s 973us/step - loss: 0.3099 - accuracy: 0.8723 - val_loss: 0.3721 - val_accuracy: 0.8555
Epoch 6/100
250/250 [==============================] - 0s 992us/step - loss: 0.3097 - accuracy: 0.8734 - val_loss: 0.3701 - val_accuracy: 0.8500
Epoch 7/100
250/250 [==============================] - 0s 949us/step - loss: 0.3055 - accuracy: 0.8751 - val_loss: 0.3743 - val_accu

In [ ]:
#Save the model
model.save('model.h5')

In [60]:
# Load tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [62]:
# Load
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 43078), started 0:01:40 ago. (Use '!kill 43078' to kill it.)